<a href="https://colab.research.google.com/github/ai-agent-yonsei/agent/blob/main/7_3_Multi_Agent_Framework_II_Chapter3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Chapter 3. Reliability & Operations




| 실습 | 주제 |  
|---|---|
| 0 | 공통 설정 (MOCK/REAL, 시크릿, 호출 기록) |  
| 3-1 | 실패 분류와 증상→원인→장치 진단 |
| 3-2 | 런타임 감시 (결정적 모니터) |
| 3-3 | 개입 디버깅 |  
| 3-4 | 관측 가능성: 트레이스와 다섯 축 |
| 3-5 | 비용·지연 모델과 손잡이 |  
| 3-6 | 배포 전 체크리스트 (일곱 원칙) |  


In [ ]:
# ===== 공통 설정 (실습 0) =====
# 필요한 패키지를 설치함. 코랩에는 대부분 이미 있어서 금방 끝남
%pip install -q langgraph openai scikit-learn pandas

import os, re, json, time, operator, random
from typing import Annotated, TypedDict, Literal
import pandas as pd

# ---- 실행 모드 스위치 ----
#USE_REAL_LLM = False      # False=MOCK(모의 응답, 비용 0)
USE_REAL_LLM = True      #  True=OpenAI API 실제 호출임
MODEL = "gpt-4o-mini"     # 실제 호출 시 쓸 모델명임. 계정에서 쓸 수 있는 모델로 바꿔도 됨

# ---- OpenAI 클라이언트 (코랩 시크릿 사용) ----
client = None
if USE_REAL_LLM:
    from google.colab import userdata          # 코랩 시크릿을 읽는 모듈임
    from openai import OpenAI
    # 왼쪽 열쇠(시크릿) 메뉴에 OPENAI_API_KEY를 등록하고 노트북 액세스를 켜야 함
    client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

# ---- 호출 기록 ----
CALL_LOG = []             # 모든 LLM 호출을 여기에 쌓음

def est_tokens(text):
    # MOCK용 토큰 어림 계산임. 글자 수의 절반으로 가정함 (대략적인 값임)
    return max(1, round(len(text) / 2))

def call_llm(system, user, mock, tag="llm", json_mode=False):
    """LLM 호출 공통 함수임. MOCK이면 mock 값을, REAL이면 OpenAI 응답을 반환함"""
    t0 = time.time()
    if USE_REAL_LLM:
        # json_mode면 JSON 객체로만 답하게 강제함 (프롬프트에 'JSON' 단어가 있어야 함)
        kwargs = {"response_format": {"type": "json_object"}} if json_mode else {}
        resp = client.chat.completions.create(
            model=MODEL, temperature=0,
            messages=[{"role": "system", "content": system},
                      {"role": "user", "content": user}], **kwargs)
        text = resp.choices[0].message.content
        in_tok, out_tok = resp.usage.prompt_tokens, resp.usage.completion_tokens
    else:
        # MOCK: API를 부르지 않고 미리 정한 응답을 씀. 호출은 똑같이 기록함
        text = mock() if callable(mock) else mock
        in_tok, out_tok = est_tokens(system + user), est_tokens(text)
    CALL_LOG.append({"tag": tag, "in_tok": in_tok, "out_tok": out_tok,
                     "sec": round(time.time() - t0, 3)})
    return text

def reset_log():
    CALL_LOG.clear()          # 실험마다 기록을 비우고 시작함

def log_summary():
    # 호출 수와 토큰 합계를 돌려줌
    return {"calls": len(CALL_LOG),
            "in_tok": sum(c["in_tok"] for c in CALL_LOG),
            "out_tok": sum(c["out_tok"] for c in CALL_LOG)}

print("모드:", "REAL (OpenAI 호출)" if USE_REAL_LLM else "MOCK (모의 응답)")

모드: REAL (OpenAI 호출)


### 코드별 필요성과 의미

| 코드 | 왜 필요한가 | 어떤 의미인가 |
|---|---|---|
| `USE_REAL_LLM` 스위치 | 비용 없이 먼저 흐름을 익히고, 필요할 때만 실제 API를 쓰려고 둠 | MOCK은 API를 부르지 않고 미리 정한 응답으로 호출 횟수만 똑같이 세는 방식임 |
| `userdata.get(...)` | API 키를 코드에 직접 쓰면 유출 위험이 있어서 코랩 시크릿에서 읽음 | 키가 노트북 파일에 남지 않음 |
| `call_llm()` | 모든 LLM 호출을 한 함수로 모아 호출 수·토큰을 한곳에서 셈 | Chapter 3의 비용·관측 논의가 이 기록 위에서 성립함 |
| `est_tokens()` | MOCK에는 실제 토큰 수가 없어서 어림값이 필요함 | 절대값보다 줄어드는 방향과 비율을 보는 용도임 |


---
## 실습 3-1. 실패 분류(MAST 관점)와 증상→원인→장치 진단

- **목표**: 실패 트레이스 4종을 만들고, 규칙으로 증상을 탐지해 범주·원인·해결 장치를 붙이는 진단 표를 만듦

In [ ]:
from collections import Counter

# ---- 실패 트레이스 4종 + 정상 1종을 손으로 만듦 (이벤트 목록임) ----
def trace_loop():
    # 종료 조건이 없어 같은 호출을 계속함
    return [{"agent": "supervisor", "type": "call", "target": "account"} for _ in range(6)]

def trace_pingpong():
    # 서로에게 핸드오프만 반복함
    return [{"agent": a, "type": "handoff", "target": b} for _ in range(3) for a, b in [("account", "product"), ("product", "account")]]

def trace_missing_info():
    # 필수 정보 없이 넘겨서 전문가가 못 함
    return [{"agent": "triage", "type": "handoff", "target": "account", "packet": {"request": "잔액"}},
            {"agent": "account", "type": "answer", "text": "정보 없음: 고객번호를 알 수 없음"}]

def trace_incomplete():
    # 의도는 2개인데 1개만 처리하고 끝냄
    return [{"agent": "router", "type": "route", "labels": ["account"], "intents": ["account", "product"]},
            {"agent": "account", "type": "answer", "text": "잔액 안내"},
            {"agent": "system", "type": "finish"}]

def trace_ok():
    return [{"agent": "router", "type": "route", "labels": ["account", "product"], "intents": ["account", "product"]},
            {"agent": "account", "type": "answer", "text": "잔액 안내"},
            {"agent": "product", "type": "answer", "text": "금리 안내"},
            {"agent": "system", "type": "finish"}]

In [ ]:
# ---- 증상 탐지 규칙 ----
def diagnose(trace):
    found = []
    # 루프: 같은 (에이전트, 종류, 대상) 호출이 4회 이상임
    cnt = Counter((e["agent"], e["type"], e.get("target")) for e in trace)
    if any(n >= 4 for n in cnt.values()) and not any(e["type"] == "handoff" for e in trace):
        found.append("loop")
    # 핑퐁: A→B 핸드오프와 B→A 핸드오프가 모두 있음
    hs = {(e["agent"], e["target"]) for e in trace if e["type"] == "handoff"}
    if any((b, a) in hs for a, b in hs):
        found.append("pingpong")
    # 정보 누락: 전문가가 정보 없음이라고 답함
    if any("정보 없음" in e.get("text", "") for e in trace):
        found.append("missing_info")
    # 불완전: 의도 중 라우팅되지 않은 것이 있는데 종료함
    for e in trace:
        if e["type"] == "route" and set(e["intents"]) - set(e["labels"]):
            found.append("incomplete")
    return found or ["정상"]

# 증상 → (MAST 범주, 원인, 장치) 대응표
FIXES = {
    "loop":         ("① 시스템 설계·명세", "종료 조건 부재", "종료를 겹으로 + 진전 없음 모니터 (실습 1-4, 3-2)"),
    "pingpong":     ("① 시스템 설계·명세", "역할 경계 모호", "배타적 역할 + 상한 + 에스컬레이션 (실습 2-5)"),
    "missing_info": ("② 에이전트 간 불일치", "정보가 전달되지 않음", "structured handoff + 계약 (실습 2-3, 2-4)"),
    "incomplete":   ("③ 과제 검증", "불완전한 검증", "multi-label fan-out + 완료 검증 (실습 1-7)"),
}

rows = []
for name, tr in [("루프", trace_loop()), ("핑퐁", trace_pingpong()), ("정보 누락", trace_missing_info()),
                 ("일부 누락", trace_incomplete()), ("정상", trace_ok())]:
    for sym in diagnose(tr):
        cat, cause, fix = FIXES.get(sym, ("-", "-", "-"))
        rows.append({"사례": name, "탐지된 증상": sym, "범주": cat, "원인": cause, "장치": fix})
pd.DataFrame(rows)

,사례,탐지된 증상,범주,원인,장치
0,루프,loop,① 시스템 설계·명세,종료 조건 부재,"종료를 겹으로 + 진전 없음 모니터 (실습 1-4, 3-2)"
1,핑퐁,pingpong,① 시스템 설계·명세,역할 경계 모호,배타적 역할 + 상한 + 에스컬레이션 (실습 2-5)
2,정보 누락,missing_info,② 에이전트 간 불일치,정보가 전달되지 않음,"structured handoff + 계약 (실습 2-3, 2-4)"
3,일부 누락,incomplete,③ 과제 검증,불완전한 검증,multi-label fan-out + 완료 검증 (실습 1-7)
4,정상,정상,-,-,-


### 코드별 필요성과 의미

| 코드 | 왜 필요한가 | 어떤 의미인가 |
|---|---|---|
| 트레이스 4종 + 정상 | 실패는 이름을 붙이려면 먼저 재현 가능한 기록이 있어야 함 | 정상 사례가 있어야 오탐(false positive)이 없는지 확인할 수 있음 |
| `diagnose()` 규칙 | 사람이 매번 로그를 읽는 대신 증상을 자동으로 분류하려고 만듦 | 반복·순환·메시지·완료 여부 같은 구조적 단서로 잡는 가벼운 진단임 |
| `FIXES` 대응표 | 증상만 알면 고칠 곳을 모름 | 증상 → 원인 → 장치를 연결하면 어느 Chapter의 도구를 쓸지 바로 정해짐 |
| MAST 범주 열 | 실패를 설계·소통·검증으로 나누면 책임 영역이 보임 | MAST의 3개 범주 관점임 (범주 이름은 슬라이드 28을 따름) |

**관찰 포인트**: 정상 사례만 증상 없음으로 나오고, 나머지 4건은 각각 다른 범주와 장치로 연결됨. 핑퐁 사례는 loop로 잘못 잡히지 않도록 핸드오프 여부를 함께 봄

---
## 실습 3-2. 런타임 감시: LLM 없는 결정적 모니터

- **목표**: 같은 오류를 반복하는 에이전트를 모니터 없이/있이 돌려 호출 수와 토큰을 비교함

In [ ]:
# 환경임: search_v1은 항상 타임아웃이고 search_v2는 정상 동작함
def run_tool(tool):
    if tool == "search_v1":
        return {"ok": False, "error": "TimeoutError"}
    return {"ok": True, "result": "예금 금리 3.1%"}

def agent_step(history, hint):
    # 에이전트(LLM)가 다음 도구를 고름. MOCK은 힌트가 없으면 v1을 고집하고, 힌트가 있으면 v2로 바꿈
    tool = "search_v2" if hint else "search_v1"
    out = call_llm(system='에이전트임. 사용할 도구를 JSON {"tool": ...}로만 답함',
                   user=json.dumps({"최근기록": history[-3:], "힌트": hint}, ensure_ascii=False),
                   mock=json.dumps({"tool": tool}), tag="agent", json_mode=True)
    return json.loads(out)["tool"]

class Monitor:
    """LLM을 쓰지 않는 규칙 기반 감시자임"""
    def __init__(self, repeat_limit=2):
        self.recent, self.limit = [], repeat_limit
    def observe(self, tool, result):
        self.recent.append((tool, result.get("error")))
    def check(self):
        # 같은 도구가 같은 오류로 limit번 연속 실패하면 개입 메시지를 만듦
        last = self.recent[-self.limit:]
        if len(last) == self.limit and len(set(last)) == 1 and last[0][1]:
            return f"{last[0][0]}가 {last[0][1]}로 반복 실패함. 다른 도구를 사용할 것"
        return None

In [ ]:
def run_episode(use_monitor, max_steps=15):
    reset_log()
    history, hint, mon = [], None, Monitor()
    for step in range(1, max_steps + 1):
        tool = agent_step(history, hint)
        res = run_tool(tool)
        history.append({"tool": tool, "결과": res})
        if res["ok"]:
            return {"성공": True, "단계": step, **log_summary()}
        if use_monitor:                                  # 모니터는 매 단계 결과만 보고 판단함 (LLM 호출 없음)
            mon.observe(tool, res)
            hint = mon.check() or hint
    return {"성공": False, "단계": max_steps, **log_summary()}

EP_BROKEN = run_episode(use_monitor=False)
EP_MON = run_episode(use_monitor=True)
df32 = pd.DataFrame([{"구분": "모니터 없음", **EP_BROKEN}, {"구분": "모니터 적용", **EP_MON}]).rename(
    columns={"calls": "LLM 호출", "in_tok": "입력 토큰", "out_tok": "출력 토큰"})
df32

,구분,성공,단계,LLM 호출,입력 토큰,출력 토큰
0,모니터 없음,True,1,1,43,22
1,모니터 적용,True,1,1,43,22


### 코드별 필요성과 의미

| 코드 | 왜 필요한가 | 어떤 의미인가 |
|---|---|---|
| `run_tool()`의 고장난 도구 | 반복 실패 상황을 의도적으로 만들어야 감시 효과를 볼 수 있음 | 실제 운영에서 흔한 타임아웃·재시도 루프를 흉내 낸 것임 |
| `Monitor.check()` | LLM으로 감시하면 감시 비용이 또 생김 | 오류·반복 패턴을 규칙으로 잡아서 감시 비용이 거의 0임 |
| `hint` 주입 | 감지만 하고 개입하지 않으면 낭비가 안 멈춤 | 필요할 때만 개입하고 기존 에이전트 구조는 바꾸지 않음 |
| `max_steps` 비교 | 모니터가 없을 때의 낭비를 기준선으로 봐야 함 | 호출·토큰 차이가 곧 모니터의 효과임 |

모니터가 없으면 max_steps까지 실패를 반복하고 결과가 없음. 모니터가 있으면 2번 실패한 뒤 개입해 3번째 호출에서 성공함. REAL 모드에서는 LLM이 힌트를 다르게 해석할 수 있어 MOCK에서 먼저 확인하는 것을 권장함

---
## 실습 3-3. 개입 디버깅: 가설 → 개입 → 재실행 → 검증

- **목표**: 실패하는 파이프라인에서 가설 3개를 하나씩 개입으로 시험해 진짜 원인만 남김

In [ ]:
# 의도별 단어 사전임 (앞 노트북과 같은 구성임)
INTENT_WORDS = {
    "account": ["잔액", "이체", "거래 내역", "비밀번호", "통장", "계좌"],
    "product": ["금리", "가입", "상품", "대출", "적금", "예금", "펀드", "연금"],
    "care":    ["항의", "민원", "불친절", "피해", "환불", "부당", "해명"],
}
def keyword_labels(text):
    return [k for k, ws in INTENT_WORDS.items() if any(w in text for w in ws)]

CID_PATTERN = r"C-\d{4}"                # 고객번호 형태(C-숫자4자리)임
LONG_HISTORY = [
    "고객: 안녕하세요, 고객번호는 C-1042입니다.",
    "상담원: 무엇을 도와드릴까요?",
    "고객: 지난달부터 계좌에서 돈이 줄었어요.",
    "고객: 잔액 좀 확인해 주세요.",
]

# 파이프라인임. 설정(cfg) 3개를 바꿔 가며 재실행할 수 있음
def run_pipeline(cfg):
    trace = []
    # 1) 라우팅: route_override가 있으면 강제로 고정함
    route = cfg.get("route_override") or (keyword_labels(LONG_HISTORY[-1]) or ["care"])[0]
    trace.append(f"route={route}")
    # 2) 핸드오프 맥락 구성: last면 마지막 발화만, structured면 필수 필드를 함께 넘김
    if cfg["handoff_mode"] == "structured":
        cid = re.search(CID_PATTERN, "\n".join(LONG_HISTORY)).group(0)
        ctx = json.dumps({"customer_id": cid}, ensure_ascii=False) + "\n" + LONG_HISTORY[-1]
    else:
        ctx = LONG_HISTORY[-1]
    trace.append(f"context_has_id={bool(re.search(CID_PATTERN, ctx))}")
    # 3) 도구 권한 확인
    allowed = cfg["tool_perm"].get(route, [])
    trace.append(f"tool_allowed={'get_balance' in allowed}")
    # 4) 전문가 실행: 고객번호가 보이고 도구 권한이 있어야 성공함
    ok = route == "account" and "get_balance" in allowed and "C-1042" in ctx
    return {"성공": ok, "trace": " | ".join(trace)}

BASELINE = {"route_override": None, "handoff_mode": "last", "tool_perm": {"account": ["get_balance"]}}
print("기준 실행:", run_pipeline(BASELINE))

기준 실행: {'성공': False, 'trace': 'route=account | context_has_id=False | tool_allowed=True'}


In [ ]:
# 가설마다 '개입'을 하나만 적용해서 다시 돌려 봄 (개입 외에는 기준 설정 그대로임)
hypotheses = [
    ("H1 라우팅이 틀렸다",   {**BASELINE, "route_override": "account"}),
    ("H2 도구 권한이 없다",  {**BASELINE, "tool_perm": {"account": ["get_balance", "get_history", "transfer"]}}),
    ("H3 맥락이 부족하다",   {**BASELINE, "handoff_mode": "structured"}),
]
base = run_pipeline(BASELINE)["성공"]
rows = []
for name, cfg in hypotheses:
    r = run_pipeline(cfg)
    verdict = "검증됨 (원인)" if (r["성공"] and not base) else "반박됨"
    rows.append({"가설": name, "기준 결과": "성공" if base else "실패",
                 "개입 후 결과": "성공" if r["성공"] else "실패", "판정": verdict})
pd.DataFrame(rows)

,가설,기준 결과,개입 후 결과,판정
0,H1 라우팅이 틀렸다,실패,실패,반박됨
1,H2 도구 권한이 없다,실패,실패,반박됨
2,H3 맥락이 부족하다,실패,성공,검증됨 (원인)


### 코드별 필요성과 의미

| 코드 | 왜 필요한가 | 어떤 의미인가 |
|---|---|---|
| `run_pipeline(cfg)` | 개입을 바꿔 가며 같은 조건으로 재실행하려면 설정으로 조절 가능해야 함 | 실행 기록을 읽는 것(관측)과 고쳐서 다시 돌리는 것(개입)은 다른 일임 |
| `trace` 문자열 | 무엇이 실제로 일어났는지 근거가 있어야 가설을 세울 수 있음 | route·context·권한 3개 지점이 의심 후보가 됨 |
| 가설별 단일 개입 | 한 번에 여러 개를 고치면 무엇이 효과였는지 모름 | 하나만 바꿔서 결과가 뒤집히는지 보는 것이 인과 확인임 |
| 판정 로직 | 추측을 사실로 오해하지 않으려고 둠 | 고쳐서 결과가 뒤집힌 가설만 진짜 원인으로 인정함 |

H1·H2는 고쳐도 계속 실패해서 반박되고, H3(맥락 부족)만 성공으로 뒤집혀 검증됨. 트레이스만 봤다면 라우팅이나 권한을 의심했을 수 있음

---
## 실습 3-4. 관측 가능성: 호출마다 트레이스를 남기고 다섯 축으로 요약

- **목표**: 에이전트 호출마다 같은 필드를 기록하는 트레이스 장치를 만들고, 품질·비용·지연·조율·안전 요약을 계산함

In [ ]:
from dataclasses import dataclass, asdict

# 호출 1회에 남길 필드임 (슬라이드 32의 TRACE 필드와 같은 구성임)
@dataclass
class TraceRec:
    run: str
    stage: int              # 같은 stage는 병렬로 돌 수 있는 단계임
    agent: str
    in_tok: int
    out_tok: int
    latency: float
    route: str              # 어디로 보냈나
    handoff_context: str    # 무엇을 넘겼나
    state_diff: str         # 상태가 무엇이 바뀌었나
    stop_reason: str        # 왜 멈췄나 (done / handoff / blocked 등)

TRACE = []

def traced(run, stage, agent, fn, route="", handoff_context="", stop_reason="done"):
    """에이전트 실행을 감싸서 호출 전후의 토큰·지연·상태 변화를 기록함"""
    n0, t0 = len(CALL_LOG), time.time()
    upd = fn()
    new_calls = CALL_LOG[n0:]
    TRACE.append(TraceRec(run, stage, agent,
                          sum(c["in_tok"] for c in new_calls), sum(c["out_tok"] for c in new_calls),
                          round(time.time() - t0, 3), route, handoff_context,
                          json.dumps(upd, ensure_ascii=False)[:60], stop_reason))
    return upd

def fake_agent(name, text, delay):
    # 일하는 척하는 에이전트임: delay초 걸리고 LLM을 1번 부름
    def fn():
        time.sleep(delay)
        call_llm(system=f"{name} 담당임", user="요청 처리", mock=text, tag=name)
        return {name + "_answer": text}
    return fn

def run_pipeline(run_id):
    traced(run_id, 0, "triage", fake_agent("triage", "account+product로 분배", 0.05),
           route="account,product", handoff_context="customer_id,request", stop_reason="handoff")
    traced(run_id, 1, "account", fake_agent("account", "잔액 안내", 0.12), stop_reason="done")
    traced(run_id, 1, "product", fake_agent("product", "금리 안내", 0.20), stop_reason="done")
    traced(run_id, 2, "synth", fake_agent("synth", "잔액 안내 + 금리 안내", 0.05), stop_reason="done")
    return "잔액 안내 + 금리 안내"

reset_log(); TRACE.clear()
final_answer = run_pipeline("run-1")
pd.DataFrame([asdict(r) for r in TRACE])

,run,stage,agent,in_tok,out_tok,latency,route,handoff_context,state_diff,stop_reason
0,run-1,0,triage,18,24,0.728,"account,product","customer_id,request","{""triage_answer"": ""account+product로 분배""}",handoff
1,run-1,1,account,17,33,1.312,,,"{""account_answer"": ""잔액 안내""}",done
2,run-1,1,product,17,28,0.751,,,"{""product_answer"": ""금리 안내""}",done
3,run-1,2,synth,18,33,0.822,,,"{""synth_answer"": ""잔액 안내 + 금리 안내""}",done


In [ ]:
# 다섯 축 요약임. 트레이스 기록만으로 모두 계산함
def evaluate(answer, gold_intents):
    # 품질: 정답 의도(키워드)가 답에 모두 들어갔는지 점수화함 (0~1)
    cover = {"account": "잔액", "product": "금리", "care": "민원"}
    return sum(cover[g] in answer for g in gold_intents) / len(gold_intents)

def five_axes(trace, answer, gold_intents):
    tot_tok = sum(r.in_tok + r.out_tok for r in trace)
    score = evaluate(answer, gold_intents)
    # 지연: 합계 vs 임계 경로(단계별 최댓값의 합)임
    latency_sum = sum(r.latency for r in trace)
    critical = sum(max(r.latency for r in trace if r.stage == s) for s in {r.stage for r in trace})
    return {
        "품질(점수)": round(score, 2),
        "비용(호출 수 / 토큰)": f"{len(trace)}회 / {tot_tok}",
        "비용(점수 per 1k 토큰)": round(score / tot_tok * 1000, 2),
        "지연(합계 vs 임계 경로, 초)": f"{latency_sum:.2f} vs {critical:.2f}",
        "조율(핸드오프 수)": sum(r.stop_reason == "handoff" for r in trace),
        "안전(차단 수)": sum(r.stop_reason == "blocked" for r in trace),
    }

summary = five_axes(TRACE, final_answer, ["account", "product"])
pd.DataFrame([summary]).T.rename(columns={0: "값"})

,값
품질(점수),1.0
비용(호출 수 / 토큰),4회 / 188
비용(점수 per 1k 토큰),5.32
"지연(합계 vs 임계 경로, 초)",3.61 vs 2.86
조율(핸드오프 수),1
안전(차단 수),0


In [ ]:
# JSON Lines로 내보내면 어떤 로그 도구로도 넘길 수 있음 (필드를 같게 유지하는 것이 핵심임)
for r in TRACE[:2]:
    print(json.dumps(asdict(r), ensure_ascii=False))

{"run": "run-1", "stage": 0, "agent": "triage", "in_tok": 18, "out_tok": 24, "latency": 0.728, "route": "account,product", "handoff_context": "customer_id,request", "state_diff": "{\"triage_answer\": \"account+product로 분배\"}", "stop_reason": "handoff"}
{"run": "run-1", "stage": 1, "agent": "account", "in_tok": 17, "out_tok": 33, "latency": 1.312, "route": "", "handoff_context": "", "state_diff": "{\"account_answer\": \"잔액 안내\"}", "stop_reason": "done"}


### 코드별 필요성과 의미

| 코드 | 왜 필요한가 | 어떤 의미인가 |
|---|---|---|
| `TraceRec` 필드 | 무엇이 있었는지와 왜 비용이 들었는지를 둘 다 설명해야 함 | 도구가 달라도 같은 필드를 남기는 것이 핵심임 |
| `traced()` 래퍼 | 에이전트마다 기록 코드를 넣으면 빠지거나 달라짐 | 한 곳에서 모든 호출을 같은 방식으로 기록함 |
| `stage` | 병렬 가능한 단계를 알아야 임계 경로를 계산할 수 있음 | 같은 stage는 동시에 돌 수 있는 호출임 |
| `five_axes()` | 품질만 보면 비용 폭증이나 위험 신호를 놓침 | 품질·비용·지연·조율·안전을 한 표에서 봄 |
| `score per 1k tokens` | 점수와 비용을 한 지표로 묶어 비교하려고 둠 | 같은 점수라면 싼 쪽이 좋은 설계임 |

지연의 합계(0.42초)보다 임계 경로(0.30초)가 짧음. account와 product가 같은 stage라서 병렬이면 느린 쪽(0.20초)만 기다리면 되고, 임계 경로는 0.05 + 0.20 + 0.05임

---
## 실습 3-5. 비용·지연 모델과 줄이는 손잡이


- **목표**: 비용 식과 지연 식을 코드로 만들고, 손잡이별 절감 효과를 같은 표에서 비교함

In [ ]:
# 단가는 예시값임 (100만 토큰당 USD). 실제 단가는 OpenAI 가격표를 확인해서 바꿔 넣을 것
P_IN, P_OUT = 0.15, 0.60

def cost_usd(in_tok, out_tok):
    # 비용 ≈ Σ 호출 × (입력 × p_in + 출력 × p_out) 임. 호출별 토큰을 합쳐 넣으면 됨
    return in_tok / 1e6 * P_IN + out_tok / 1e6 * P_OUT

print("예: 입력 1,000토큰 + 출력 200토큰 1회 =", f"${cost_usd(1000, 200):.6f}")

예: 입력 1,000토큰 + 출력 200토큰 1회 = $0.000270


In [ ]:
from concurrent.futures import ThreadPoolExecutor

# ---- 지연: 병렬 실행의 실제 시간 vs 합계 ----
def job(sec):
    time.sleep(sec); return sec

stage1 = [0.4, 0.3, 0.5]                      # 병렬로 돌릴 3개 호출의 소요 시간임
t0 = time.time()
with ThreadPoolExecutor() as ex:
    list(ex.map(job, stage1))
parallel_wall = time.time() - t0
print(f"순차 합계 {sum(stage1):.2f}초 vs 병렬 실제 {parallel_wall:.2f}초 (임계 경로 = 가장 느린 {max(stage1):.2f}초)")

순차 합계 1.20초 vs 병렬 실제 0.51초 (임계 경로 = 가장 느린 0.50초)


In [ ]:
# ---- 손잡이별 비교표 ----
# (1) 라우팅: 호출 수. 실습 1-6 결과를 직접 넣을 것 (예시값: LLM 단독 18회 vs cascade 4회)
CH1_LLM_ONLY_CALLS, CH1_CASCADE_CALLS = 18, 4

# (2) 맥락 범위: 같은 이력을 full vs structured로 넘길 때의 입력 토큰
HIST = ["고객: 안녕하세요, 고객번호는 C-1042입니다.", "상담원: 무엇을 도와드릴까요?",
        "고객: 날씨가 덥네요.", "상담원: 네, 문의 내용을 말씀해 주세요.",
        "고객: 지난달부터 계좌에서 돈이 줄었어요.", "고객: 잔액 좀 확인해 주세요."]
full_tok = est_tokens("\n".join(HIST))
structured_tok = est_tokens('{"customer_id": "C-1042"}\n' + HIST[-1])

# (3) 종료·감시: 실습 3-2의 결과 (없으면 예시값 사용)
try:
    broken_tok = EP_BROKEN["in_tok"] + EP_BROKEN["out_tok"]
    monitored_tok = EP_MON["in_tok"] + EP_MON["out_tok"]
except NameError:
    broken_tok, monitored_tok = 2293, 1419       # 슬라이드의 예시 수치임

# (4) 격리: 세 번째 워커 입력 (실습 2-7의 MOCK 어림값을 가정한 예시값임)
shared_tok3, isolated_tok3 = 343, 88

levers = [
    ("라우팅", "호출 수", CH1_LLM_ONLY_CALLS, CH1_CASCADE_CALLS, "회"),
    ("맥락 범위", "입력 토큰", full_tok, structured_tok, "토큰"),
    ("격리", "누적 입력 토큰", shared_tok3, isolated_tok3, "토큰"),
    ("종료·감시", "루프 낭비 토큰", broken_tok, monitored_tok, "토큰"),
    ("병렬화", "지연", round(sum(stage1), 2), round(max(stage1), 2), "초"),
]
df35 = pd.DataFrame(levers, columns=["손잡이", "줄이는 대상", "전", "후", "단위"])
df35["절감률"] = ((1 - df35["후"] / df35["전"]) * 100).round(0).astype(int).astype(str) + "%"
df35["전"] = df35["전"].map(lambda x: f"{x:g}")        # 18.0 대신 18로 보기 좋게 바꿈
df35["후"] = df35["후"].map(lambda x: f"{x:g}")
df35

,손잡이,줄이는 대상,전,후,단위,절감률
0,라우팅,호출 수,18,4,회,78%
1,맥락 범위,입력 토큰,62,22,토큰,65%
2,격리,누적 입력 토큰,343,88,토큰,74%
3,종료·감시,루프 낭비 토큰,65,65,토큰,0%
4,병렬화,지연,1.2,0.5,초,58%


### 코드별 필요성과 의미

| 코드 | 왜 필요한가 | 어떤 의미인가 |
|---|---|---|
| `cost_usd()` | 토큰 수를 돈으로 바꿔야 의사결정에 쓸 수 있음 | 입력 단가와 출력 단가가 달라서 따로 곱함. 단가는 예시값이므로 최신 가격표로 교체해야 함 |
| `ThreadPoolExecutor` 병렬 측정 | 지연은 합계가 아니라 가장 느린 경로가 정함을 실측으로 보이려고 씀 | 병렬화의 이득은 임계 경로 기준으로 계산함 |
| `levers` 표 | 어느 손잡이가 얼마나 줄이는지 한눈에 비교해야 우선순위를 정함 | 라우팅·맥락·격리·감시·병렬화가 앞선 실습과 일대일로 대응함 |
| `try/except NameError` | 3-2를 건너뛰고 실행해도 표가 깨지지 않게 하려고 둠 | 없으면 예시 수치로 대체함. 가능하면 3-2를 먼저 실행할 것 |

병렬 실제 시간은 순차 합계보다 가장 느린 호출 수준으로 줄어듦. 표의 일부 값(라우팅, 격리)은 예시 입력이므로 앞 실습의 자기 결과로 바꿔 넣으면 더 정확함

---
## 실습 3-6. 배포 전 체크리스트: 일곱 가지 설계 원칙 자동 점검

- **목표**: 시스템 설정(cfg)을 입력받아 일곱 원칙을 통과하는지 점검하고, 미흡한 설정과 개선한 설정을 비교함

In [ ]:
REQUIRED_TRACE_FIELDS = {"agent", "in_tok", "out_tok", "latency", "route", "handoff_context", "state_diff", "stop_reason"}

def check_deploy(cfg):
    checks = [
        ("1 정적이 기본",
         cfg["static_backbone"] and cfg["dynamic_decisions"] <= 2,
         "정적 뼈대가 있고 동적 결정이 2곳 이하여야 함"),
        ("2 종료를 겹으로",
         len(cfg["termination_layers"]) >= 3 and "recursion_limit" in cfg["termination_layers"],
         "종료 장치가 3겹 이상이고 recursion_limit은 마지막 안전망으로 포함돼야 함"),
        ("3 싸게 먼저",
         cfg["router_order"][:1] == ["rule"] and cfg["router_order"][-1] == "llm",
         "라우터가 규칙으로 시작해 LLM으로 끝나야 함"),
        ("4 handoff = control + context",
         cfg["handoff_structured"] and cfg["contracts"],
         "핸드오프가 구조화 패킷이고 계약 검사가 있어야 함"),
        ("5 허용 에지로만",
         cfg["allowed_edges"] and cfg["code_approves_goto"],
         "허용 에지가 정의돼 있고 goto 승인을 코드가 해야 함"),
        ("6 상태는 키별 규칙",
         all(v in ("reducer", "single_writer") for v in cfg["state_keys"].values()),
         "모든 상태 키가 reducer 또는 single writer여야 함"),
        ("7 실패를 재현",
         REQUIRED_TRACE_FIELDS <= set(cfg["trace_fields"]) and cfg["intervention_tests"],
         "필수 트레이스 필드를 모두 남기고 개입 재현 테스트가 있어야 함"),
    ]
    return pd.DataFrame([{"원칙": n, "판정": "통과" if ok else "미흡", "기준": why} for n, ok, why in checks])

BAD = {
    "static_backbone": False, "dynamic_decisions": 5,
    "termination_layers": ["recursion_limit"],
    "router_order": ["llm"],
    "handoff_structured": False, "contracts": False,
    "allowed_edges": False, "code_approves_goto": False,
    "state_keys": {"log": "reducer", "answer": "none", "facts": "none"},
    "trace_fields": ["agent", "in_tok"], "intervention_tests": False,
}
GOOD = {
    "static_backbone": True, "dynamic_decisions": 1,
    "termination_layers": ["goal", "max_rounds", "no_progress", "recursion_limit"],
    "router_order": ["rule", "classifier", "llm"],
    "handoff_structured": True, "contracts": True,
    "allowed_edges": True, "code_approves_goto": True,
    "state_keys": {"log": "reducer", "answer": "single_writer", "facts": "reducer"},
    "trace_fields": list(REQUIRED_TRACE_FIELDS), "intervention_tests": True,
}

In [ ]:
for name, cfg in [("미흡한 설정", BAD), ("개선한 설정", GOOD)]:
    df = check_deploy(cfg)
    print(f"=== {name}: {(df['판정'] == '통과').sum()}/7 통과 → 배포 {'가능' if (df['판정'] == '통과').all() else '보류'} ===")
    display(df)

=== 미흡한 설정: 0/7 통과 → 배포 보류 ===


,원칙,판정,기준
0,1 정적이 기본,미흡,정적 뼈대가 있고 동적 결정이 2곳 이하여야 함
1,2 종료를 겹으로,미흡,종료 장치가 3겹 이상이고 recursion_limit은 마지막 안전망으로 포함돼야 함
2,3 싸게 먼저,미흡,라우터가 규칙으로 시작해 LLM으로 끝나야 함
3,4 handoff = control + context,미흡,핸드오프가 구조화 패킷이고 계약 검사가 있어야 함
4,5 허용 에지로만,미흡,허용 에지가 정의돼 있고 goto 승인을 코드가 해야 함
5,6 상태는 키별 규칙,미흡,모든 상태 키가 reducer 또는 single writer여야 함
6,7 실패를 재현,미흡,필수 트레이스 필드를 모두 남기고 개입 재현 테스트가 있어야 함


=== 개선한 설정: 7/7 통과 → 배포 가능 ===


,원칙,판정,기준
0,1 정적이 기본,통과,정적 뼈대가 있고 동적 결정이 2곳 이하여야 함
1,2 종료를 겹으로,통과,종료 장치가 3겹 이상이고 recursion_limit은 마지막 안전망으로 포함돼야 함
2,3 싸게 먼저,통과,라우터가 규칙으로 시작해 LLM으로 끝나야 함
3,4 handoff = control + context,통과,핸드오프가 구조화 패킷이고 계약 검사가 있어야 함
4,5 허용 에지로만,통과,허용 에지가 정의돼 있고 goto 승인을 코드가 해야 함
5,6 상태는 키별 규칙,통과,모든 상태 키가 reducer 또는 single writer여야 함
6,7 실패를 재현,통과,필수 트레이스 필드를 모두 남기고 개입 재현 테스트가 있어야 함


### 코드별 필요성과 의미

| 코드 | 왜 필요한가 | 어떤 의미인가 |
|---|---|---|
| `check_deploy()` | 원칙은 외워서가 아니라 점검 코드로 지켜져야 함 | 배포 전에 자동으로 도는 게이트로 만들 수 있음 |
| `REQUIRED_TRACE_FIELDS` | 원칙 7은 기록 필드가 빠지면 재현이 불가능함 | 실습 3-4의 트레이스 필드와 같은 집합임 |
| `BAD` / `GOOD` 설정 | 미흡한 상태와 개선된 상태를 나란히 봐야 기준이 이해됨 | 각 항목이 앞선 실습의 어느 도구로 해결되는지 연결해서 볼 수 있음 |
| 모두 통과해야 배포 | 하나라도 미흡하면 운영 사고의 원인이 됨 | 슬라이드 34의 '설명할 수 있는가' 체크를 코드로 옮긴 것임 |

미흡한 설정은 대부분의 원칙에서 미흡이고, 개선한 설정은 7/7 통과함. 미흡 항목마다 해결하는 실습 번호(1-4, 1-6, 2-4, 2-6, 3-4 등)를 떠올려 볼 것